# Stockholm Stadsdelar - Demografisk Data
Hämtar data för 127 stadsdelar (RegSO) i Stockholm

In [1]:
import requests
import pandas as pd
import itertools

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
LANG = "sv"

TABLE_FOLKM_DESO = "TAB6574"
TABLE_HUSHALL    = "TAB6568"
TABLE_BOENDE     = "TAB6638"

BATCH_SIZE = 10

In [2]:
def get_metadata(table_id):
    url = f"{BASE_URL}/tables/{table_id}/metadata?lang={LANG}"
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

def stockholm_regso_koder(meta):
    """Hämta RegSO-koder för Stockholm med stadsdelsnamn"""
    region_cats = meta["dimension"]["Region"]["category"]["index"]
    region_labels = meta["dimension"]["Region"]["category"]["label"]
    
    regso = {k: region_labels[k] for k in region_cats.keys() 
             if k.startswith("0180") and k.endswith("_RegSO2025")}
    
    print(f"  {len(regso)} stadsdelar hittade")
    return regso

def jsonstat2_to_df(data, region_namn_map):
    """Konvertera JSON-stat2 till DataFrame och lägg till stadsdelsnamn"""
    dims = data["id"]
    values = data["value"]
    
    # Hämta värden för varje dimension
    dim_vals = []
    for d in dims:
        if d == "Region":
            # För Region: använd koderna (inte labels)
            dim_vals.append(list(data["dimension"][d]["category"]["index"].keys()))
        else:
            dim_vals.append(list(data["dimension"][d]["category"]["label"].values()))
    
    df = pd.DataFrame(list(itertools.product(*dim_vals)), columns=dims)
    df["value"] = values
    
    # Lägg till stadsdelsnamn
    df["Stadsdel"] = df["Region"].map(lambda x: region_namn_map.get(x, x).replace("Stockholm (", "").replace(")", ""))
    
    # Flytta Stadsdel till andra kolumnen
    cols = df.columns.tolist()
    cols.insert(1, cols.pop(cols.index("Stadsdel")))
    df = df[cols]
    
    return df

def fetch_batch(table_id, regso_map, extra_filters):
    """Hämta data i batcher om 10 regioner"""
    koder = list(regso_map.keys())
    batches = [koder[i:i + BATCH_SIZE] for i in range(0, len(koder), BATCH_SIZE)]
    print(f"Hämtar {len(koder)} stadsdelar i {len(batches)} batcher...")
    
    dfs = []
    for i, batch in enumerate(batches, 1):
        # Bygg URL
        params = [
            f"lang={LANG}",
            "outputFormat=JSON-stat2",
            f"valueCodes[Region]={','.join(batch)}",
        ]
        for dim, vals in extra_filters.items():
            encoded_vals = [v.replace("+", "%2B") for v in vals]
            params.append(f"valueCodes[{dim}]={','.join(encoded_vals)}")
        
        url = f"{BASE_URL}/tables/{table_id}/data?" + "&".join(params)
        
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                df = jsonstat2_to_df(r.json(), regso_map)
                dfs.append(df)
            else:
                print(f"  ⚠️  Batch {i}/{len(batches)} misslyckades – HTTP {r.status_code}")
        except Exception as e:
            print(f"  ⚠️  Batch {i}/{len(batches)} misslyckades – {str(e)[:50]}")
        
        if i % 5 == 0:
            print(f"  ✓ {i}/{len(batches)} batcher klara")
    
    if not dfs:
        print("❌ Ingen data hämtades.")
        return None
    
    result = pd.concat(dfs, ignore_index=True)
    print(f"✅ Klart – {len(result)} rader, {result['Stadsdel'].nunique()} stadsdelar")
    return result

print("Funktioner laddade")

Funktioner laddade


---
## 1. Folkmängd (TAB6574)
Total befolkning per stadsdel

In [3]:
print("\n" + "="*60)
print("FOLKMÄNGD")
print("="*60)

meta_folkm = get_metadata(TABLE_FOLKM_DESO)
regso_folkm = stockholm_regso_koder(meta_folkm)

df_folkm = fetch_batch(
    TABLE_FOLKM_DESO,
    regso_folkm,
    {
        "Alder": ["totalt"],
        "Kon": ["1+2"],
        "ContentsCode": ["000007Y7"],
        "Tid": ["2024"],
    }
)

if df_folkm is not None:
    # Sortera efter befolkning
    df_folkm_sorted = df_folkm.sort_values('value', ascending=False)
    
    display(df_folkm_sorted[["Stadsdel", "value"]].head(15))
    
    print(f"\n📊 Total befolkning: {df_folkm['value'].sum():,.0f}")
    print(f"   Största stadsdel: {df_folkm_sorted.iloc[0]['Stadsdel']} ({df_folkm_sorted.iloc[0]['value']:,.0f})")
    print(f"   Minsta stadsdel: {df_folkm_sorted.iloc[-1]['Stadsdel']} ({df_folkm_sorted.iloc[-1]['value']:,.0f})")


FOLKMÄNGD
  127 stadsdelar hittade
Hämtar 127 stadsdelar i 13 batcher...
  ✓ 5/13 batcher klara
  ✓ 10/13 batcher klara
✅ Klart – 127 rader, 127 stadsdelar


,Stadsdel,value
29,Gärdet,23101
123,Östra Katarina,20678
116,Årsta,20638
113,Västra Matteus,20629
93,Södra Hammarbyhamnen,20300
48,Kungsholm,19790
40,Hässelby Villastad,18974
101,Tensta,18545
52,Liljeholmen,17930
75,Rinkeby,17096



📊 Total befolkning: 995,574
   Största stadsdel: Gärdet (23,101)
   Minsta stadsdel: Djurgården (827)


---
## 2. Hushåll (TAB6568)
Totalt antal hushåll per stadsdel

In [4]:
print("\n" + "="*60)
print("HUSHÅLL")
print("="*60)

meta_hushall = get_metadata(TABLE_HUSHALL)
regso_hushall = stockholm_regso_koder(meta_hushall)

df_hushall = fetch_batch(
    TABLE_HUSHALL,
    regso_hushall,
    {
        "Hushallstyp": ["TOTALT"],
        "ContentsCode": ["000007Y1"],
        "Tid": ["2024"],
    }
)

if df_hushall is not None:
    df_hushall_sorted = df_hushall.sort_values('value', ascending=False)
    display(df_hushall_sorted[["Stadsdel", "value"]].head(15))
    
    print(f"\n🏠 Totalt hushåll: {df_hushall['value'].sum():,.0f}")


HUSHÅLL
  127 stadsdelar hittade
Hämtar 127 stadsdelar i 13 batcher...
  ✓ 5/13 batcher klara
  ✓ 10/13 batcher klara
✅ Klart – 127 rader, 127 stadsdelar


,Stadsdel,value
29,Gärdet,13510
123,Östra Katarina,11516
113,Västra Matteus,11169
48,Kungsholm,11026
116,Årsta,10892
125,Östra Sankt Göran,9493
93,Södra Hammarbyhamnen,9239
66,Norra Högalid,8634
112,Västra Katarina,8437
72,Oscars kyrka,8238



🏠 Totalt hushåll: 485,661


---
## 3. Boende (TAB6638)
Antal bostäder per upplåtelseform och stadsdel

In [5]:
print("\n" + "="*60)
print("BOENDE")
print("="*60)

meta_boende = get_metadata(TABLE_BOENDE)
regso_boende = stockholm_regso_koder(meta_boende)

df_boende = fetch_batch(
    TABLE_BOENDE,
    regso_boende,
    {
        "Upplatelseform": ["1", "2", "3", "ÖVRIGT"],
        "ContentsCode": ["00000864"],
        "Tid": ["2024"],
    }
)

if df_boende is not None:
    display(df_boende[["Stadsdel", "Upplatelseform", "value"]].head(15))
    
    print(f"\n🔑 Totalt bostäder: {df_boende['value'].sum():,.0f}")
    
    print(f"\n   Fördelning per upplåtelseform:")
    boende_sum = df_boende.groupby('Upplatelseform')['value'].sum()
    for form, antal in boende_sum.items():
        procent = (antal / boende_sum.sum()) * 100
        print(f"     {form}: {antal:,.0f} ({procent:.1f}%)")


BOENDE
  127 stadsdelar hittade
Hämtar 127 stadsdelar i 13 batcher...
  ✓ 5/13 batcher klara
  ✓ 10/13 batcher klara
✅ Klart – 508 rader, 127 stadsdelar


,Stadsdel,Upplatelseform,value
0,Abrahamsberg,hyresrätt,1173
1,Abrahamsberg,bostadsrätt,902
2,Abrahamsberg,äganderätt,0
3,Abrahamsberg,uppgift saknas,0
4,Akalla,hyresrätt,1066
5,Akalla,bostadsrätt,2882
6,Akalla,äganderätt,169
7,Akalla,uppgift saknas,0
8,Alvik,hyresrätt,366
9,Alvik,bostadsrätt,549



🔑 Totalt bostäder: 522,654

   Fördelning per upplåtelseform:
     bostadsrätt: 257,568 (49.3%)
     hyresrätt: 223,826 (42.8%)
     uppgift saknas: 1 (0.0%)
     äganderätt: 41,259 (7.9%)


---
## Sammanfattning och Export

In [6]:
print("\n" + "="*60)
print("SAMMANFATTNING - STOCKHOLMS STADSDELAR (127 RegSO)")
print("="*60)

if df_folkm is not None:
    print(f"\n📊 Folkmängd 2024:")
    print(f"   {df_folkm['Stadsdel'].nunique()} stadsdelar")
    print(f"   {df_folkm['value'].sum():,.0f} invånare totalt")
    
if df_hushall is not None:
    print(f"\n🏠 Hushåll 2024:")
    print(f"   {df_hushall['Stadsdel'].nunique()} stadsdelar")
    print(f"   {df_hushall['value'].sum():,.0f} hushåll totalt")
    
if df_boende is not None:
    print(f"\n🔑 Boende 2024:")
    print(f"   {df_boende['Stadsdel'].nunique()} stadsdelar")
    print(f"   {df_boende['value'].sum():,.0f} bostäder totalt")

print("\n" + "="*60)
print("EXPORT TILL CSV")
print("="*60)

if df_folkm is not None:
    df_folkm.to_csv('stockholm_stadsdelar_folkm_2024.csv', index=False)
    print("✓ stockholm_stadsdelar_folkm_2024.csv")

if df_hushall is not None:
    df_hushall.to_csv('stockholm_stadsdelar_hushall_2024.csv', index=False)
    print("✓ stockholm_stadsdelar_hushall_2024.csv")

if df_boende is not None:
    df_boende.to_csv('stockholm_stadsdelar_boende_2024.csv', index=False)
    print("✓ stockholm_stadsdelar_boende_2024.csv")

print("\n✅ KLART! Data för 127 stadsdelar exporterat med stadsdelsnamn.")


SAMMANFATTNING - STOCKHOLMS STADSDELAR (127 RegSO)

📊 Folkmängd 2024:
   127 stadsdelar
   995,574 invånare totalt

🏠 Hushåll 2024:
   127 stadsdelar
   485,661 hushåll totalt

🔑 Boende 2024:
   127 stadsdelar
   522,654 bostäder totalt

EXPORT TILL CSV
✓ stockholm_stadsdelar_folkm_2024.csv
✓ stockholm_stadsdelar_hushall_2024.csv
✓ stockholm_stadsdelar_boende_2024.csv

✅ KLART! Data för 127 stadsdelar exporterat med stadsdelsnamn.


---
## Bonus: Skapa en kombinerad dataset

In [7]:
if df_folkm is not None and df_hushall is not None:
    # Kombinera folkmängd och hushåll
    df_combined = df_folkm[["Region", "Stadsdel", "value"]].copy()
    df_combined.rename(columns={"value": "Befolkning"}, inplace=True)
    
    df_hushall_simple = df_hushall[["Region", "value"]].copy()
    df_hushall_simple.rename(columns={"value": "Hushall"}, inplace=True)
    
    df_combined = df_combined.merge(df_hushall_simple, on="Region")
    
    # Beräkna snitt personer per hushåll
    df_combined["PersonerPerHushall"] = (df_combined["Befolkning"] / df_combined["Hushall"]).round(2)
    
    # Sortera efter befolkning
    df_combined = df_combined.sort_values("Befolkning", ascending=False)
    
    print("\nKombinerad dataset - Topp 15 stadsdelar:")
    display(df_combined[["Stadsdel", "Befolkning", "Hushall", "PersonerPerHushall"]].head(15))
    
    # Spara
    df_combined.to_csv('stockholm_stadsdelar_kombinerad_2024.csv', index=False)
    print("\n✓ stockholm_stadsdelar_kombinerad_2024.csv")


Kombinerad dataset - Topp 15 stadsdelar:


,Stadsdel,Befolkning,Hushall,PersonerPerHushall
29,Gärdet,23101,13510,1.71
123,Östra Katarina,20678,11516,1.80
116,Årsta,20638,10892,1.89
113,Västra Matteus,20629,11169,1.85
93,Södra Hammarbyhamnen,20300,9239,2.20
48,Kungsholm,19790,11026,1.79
40,Hässelby Villastad,18974,6868,2.76
101,Tensta,18545,6490,2.86
52,Liljeholmen,17930,8161,2.20
75,Rinkeby,17096,5731,2.98



✓ stockholm_stadsdelar_kombinerad_2024.csv
